In [0]:
%pip install feature-engine

In [0]:
dbutils.library.restartPython()

#SAMPLE

In [0]:
import pandas as pd

In [0]:
df = pd.read_csv("/Volumes/workspace/machine_learning/planilhas/abt_churn.csv")
df

In [0]:
df['dtRef'].value_counts().sort_index()

In [0]:
oot = df[df['dtRef'] == df['dtRef'].max()].copy()

In [0]:
df_train = df[df['dtRef']<df['dtRef'].max()].copy()

In [0]:
features = df_train.columns[2:-1]
target = 'flagChurn'

X, y = df_train[features], df_train[target]

In [0]:
from sklearn import model_selection
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)

In [0]:
print(y_train.mean())
print(y_test.mean())

#EXPLORE

In [0]:
X_train.isna().sum().sort_values(ascending=False)

In [0]:
df_analise = X_train.copy()
df_analise[target] = y_train
sumario = df_analise.groupby(by=target).agg(["mean", "median"]).T
sumario

In [0]:
sumario['diff_abs'] = sumario[0] - sumario[1]
sumario['diff_rel'] = sumario[0] / sumario[1]
sumario.sort_values(by=['diff_rel'], ascending=False)

In [0]:
from sklearn import tree
import matplotlib.pyplot as plt

arvore = tree.DecisionTreeClassifier(random_state=42,max_depth=5)
arvore.fit(X_train, y_train)

In [0]:
plt.figure(dpi=400)
tree.plot_tree(arvore, feature_names=X_train.columns, filled=True, class_names=[str(i) for i in arvore.classes_])

In [0]:
feature_importances = (pd.Series(arvore.feature_importances_,
                                 index=X_train.columns)
                       .sort_values(ascending=False)
                       .reset_index())

feature_importances['acum.']=feature_importances[0].cumsum()
feature_importances[feature_importances['acum.'] < 0.96]

#MODIFY

In [0]:
best_features = (feature_importances[feature_importances['acum.'] < 0.96]['index'].tolist())
best_features
X_train[best_features]

In [0]:
from feature_engine import discretisation, encoding
from sklearn import pipeline

In [0]:
tree_discretisation = discretisation.DecisionTreeDiscretiser(variables=best_features, regression=False, bin_output='bin_number', cv=3)



In [0]:
#tree_discretisation.fit(X_train[best_features], y_train)
#X_train_transformado = tree_discretisation.transform(X_train[best_features])

onehot = encoding.OneHotEncoder(variables=best_features, ignore_format=True)
#onehot.fit(X_train_transformado, y_train)

#X_train_transformado = onehot.transform(X_train_transformado)
#X_train_transformado


In [0]:
#arvore_nova = tree.DecisionTreeClassifier(random_state=42)
#arvore_nova.fit(X_train_transformado, y_train)


In [0]:
#d.Series(arvore_nova.feature_importances_, index=X_train_transformado.columns).sort_values(ascending=False)

In [0]:
from sklearn import linear_model

reg = linear_model.LogisticRegression(penalty=None, random_state=42, max_iter=10000)
#reg.fit(X_train_transformado, y_train)


pipeline

In [0]:
model_pipeline = pipeline.Pipeline(steps=[
    ('Discretizar', tree_discretisation),
    ('Onehot', onehot),
    ('Model', reg)
])

model_pipeline.fit(X_train, y_train)

In [0]:
from sklearn import metrics
#y_train_predict = reg.predict(X_train_transformado)
#y_train_proba = reg.predict_proba(X_train_transformado)[:,1]

y_train_predict = model_pipeline.predict(X_train)
y_train_proba = model_pipeline.predict_proba(X_train)[:,1]


acc_train_proba = metrics.roc_auc_score(y_train, y_train_proba)
acc_train = metrics.accuracy_score(y_train, y_train_predict)
print("Acuracia treino", acc_train)
print("Curva ROC", acc_train_proba)

In [0]:
#X_test_transformado = tree_discretisation.transform(X_test[best_features])
#X_test_transformado =  onehot.transform(X_test_transformado)

y_test_predict = model_pipeline.predict(X_test)
y_test_proba = model_pipeline.predict_proba(X_test)[:,1]


acc_test_proba = metrics.roc_auc_score(y_test, y_test_proba)
acc_test = metrics.accuracy_score(y_test, y_test_predict)
print("Acuracia test", acc_test)
print("Curva ROC test", acc_test_proba)

In [0]:
#oot_transformado = tree_discretisation.transform(oot[best_features])
#oot_transformado = onehot.transform(oot_transformado)

y_oot_predict = model_pipeline.predict(oot[features])
y_oot_proba = model_pipeline.predict_proba(oot[features])[:,1]


acc_oot_proba = metrics.roc_auc_score(oot[target], y_oot_proba)
acc_oot = metrics.accuracy_score(oot[target], y_oot_predict)
print("Acuracia oot", acc_oot)
print("Curva ROC oot", acc_oot_proba)